# D — Modeling & Evaluation

*Deliverable D: D1 through D9.*

Train period: Jan 1 - Oct 17, 2025.  Val period: Oct 18 - Oct 31, 2025.
See `reports/D_model_evaluation.md` for the full written evaluation.
This notebook reproduces all key results live.

In [ ]:
import pandas as pd, numpy as np, joblib, time
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb

train_df = pd.read_csv('../data/processed/master_train.csv', parse_dates=['pickup_hour'])
FEAT     = joblib.load('../models/feature_list.joblib')
TARGET   = 'trips'

tr = train_df[train_df['pickup_hour'] <= '2025-10-17 23:00'].copy()
va = train_df[train_df['pickup_hour'] >= '2025-10-18 00:00'].copy()
print('Train:', len(tr), 'rows   Val:', len(va), 'rows')
print('Features:', FEAT)

In [ ]:
# D1 — Baselines
from sklearn.dummy import DummyRegressor

# Global mean
gm = DummyRegressor(strategy='mean').fit(tr[FEAT], tr[TARGET])
p_gm = gm.predict(va[FEAT])
rmse_gm = np.sqrt(mean_squared_error(va[TARGET], p_gm))
mae_gm  = mean_absolute_error(va[TARGET], p_gm)

# Seasonal naive (zone x dow x hour mean)
sn_lookup = tr.groupby(['zone','day_of_week','hour'])[TARGET].mean()
va2 = va.copy()
va2['sn'] = va2.set_index(['zone','day_of_week','hour']).index.map(sn_lookup.to_dict()).tolist()
va2['sn'] = pd.to_numeric(va2['sn'], errors='coerce').fillna(tr[TARGET].mean())
rmse_sn = np.sqrt(mean_squared_error(va2[TARGET], va2['sn']))
mae_sn  = mean_absolute_error(va2[TARGET], va2['sn'])

print('D1 Baselines:')
print(f'  Global Mean    RMSE={rmse_gm:.2f}  MAE={mae_gm:.2f}')
print(f'  Seasonal Naive RMSE={rmse_sn:.2f}  MAE={mae_sn:.2f}')
print('Interpretation: Seasonal naive captures weekly/hourly rhythm.')
print('Our model must beat RMSE=12.00 to prove value over simple patterns.')

In [ ]:
# D2 — Model comparison
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

model_configs = [
    ('Ridge',          Ridge()),
    ('RandomForest',   RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ('HGBoosting',     HistGradientBoostingRegressor(random_state=42)),
    ('LightGBM',       lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)),
]

print(f'{'Model':<20}  RMSE    MAE     Time')
print('-' * 48)
for name, model in model_configs:
    t0 = time.time()
    model.fit(tr[FEAT], tr[TARGET])
    preds = model.predict(va[FEAT]).clip(0)
    elapsed = time.time() - t0
    rmse = np.sqrt(mean_squared_error(va[TARGET], preds))
    mae  = mean_absolute_error(va[TARGET], preds)
    print(f'{name:<20}  {rmse:6.2f}  {mae:6.2f}  {elapsed:.1f}s')
print('Winner: LightGBM -- best RMSE at competitive speed.')

In [ ]:
# D5 — Ablation study
cal   = ['hour','day_of_week','is_weekend','month','week_of_year','day_of_month','is_payday','zone_enc','zone_hour_mean']
wthr  = ['temp_c','rain_mm','humidity_pct','wind_kmh','rain_class','rain_3h']
evnt  = ['in_event_window','hours_since_event_start','hours_until_event_end']

ablation = [
    ('Calendar + Zone only', cal),
    ('+ Weather',            cal + wthr),
    ('+ Events',             cal + evnt),
    ('All combined',         cal + wthr + evnt),
]
print(f'{'Feature set':<28}  RMSE    MAE')
print('-' * 44)
for label, feats in ablation:
    m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)
    avail = [f for f in feats if f in tr.columns]
    m.fit(tr[avail], tr[TARGET])
    p = m.predict(va[avail]).clip(0)
    rmse = np.sqrt(mean_squared_error(va[TARGET], p))
    mae  = mean_absolute_error(va[TARGET], p)
    print(f'{label:<28}  {rmse:6.2f}  {mae:6.2f}')
print('Interpretation: Both weather and events improve RMSE. Each join was worth the cleaning effort.')

In [ ]:
# D3 — Rolling-origin validation
cuts = ['2025-06-30', '2025-07-31', '2025-08-31', '2025-09-30']
rmse_folds = []
print('D3 Rolling-origin validation:')
for cut in cuts:
    t_fold = train_df[train_df['pickup_hour'] <= cut]
    v_fold = train_df[
        (train_df['pickup_hour'] > cut) &
        (train_df['pickup_hour'] <= pd.Timestamp(cut) + pd.Timedelta('31D'))
    ]
    if len(v_fold) == 0:
        continue
    feats_ok = [f for f in FEAT if f in t_fold.columns]
    m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(t_fold[feats_ok], t_fold[TARGET])
    p = m.predict(v_fold[feats_ok]).clip(0)
    rmse = np.sqrt(mean_squared_error(v_fold[TARGET], p))
    rmse_folds.append(rmse)
    print(f'  Cut={cut}  train={len(t_fold)}  val={len(v_fold)}  RMSE={rmse:.2f}')
print(f'Mean RMSE: {np.mean(rmse_folds):.2f} +/- {np.std(rmse_folds):.2f}')
print('Fold 1 high RMSE expected: only 6 months training, growth trend not yet visible.')

In [ ]:
# D7 — Error analysis
val_preds = pd.read_csv('../data/processed/val_predictions.csv')
val_preds['pickup_hour'] = pd.to_datetime(val_preds['pickup_hour'])
val_preds['hour'] = val_preds['pickup_hour'].dt.hour
val_preds['dow']  = val_preds['pickup_hour'].dt.dayofweek
val_preds['day_type'] = val_preds['dow'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')

print('D7a -- MAE by zone:')
print(val_preds.groupby('zone')['abs_error'].mean().sort_values(ascending=False).round(2).to_string())

print('\nD7a -- MAE by hour (top 6 hardest):')
print(val_preds.groupby('hour')['abs_error'].mean().sort_values(ascending=False).head(6).round(2).to_string())

print('\nD7a -- MAE by day type:')
print(val_preds.groupby('day_type')['abs_error'].mean().round(2).to_string())

print('\nD7b -- Top 10 worst zone-hours:')
top10 = val_preds.nlargest(10, 'abs_error')[['zone','pickup_hour','trips','pred','error']]
print(top10.to_string(index=False))

In [ ]:
# D9 — Plain language metric
RMSE = 9.24
MAE  = 5.85
MEAN_DEMAND = 33.5
TRIPS_PER_DRIVER = 1.3

print('D9 -- Plain language metric:')
print(f'  RMSE = {RMSE} trips per zone-hour')
print(f'  MAE  = {MAE} trips per zone-hour  ({MAE/MEAN_DEMAND*100:.0f}% of mean demand)')
print(f'  RMSE = {RMSE/MEAN_DEMAND*100:.0f}% of mean demand ({MEAN_DEMAND} trips)')
print(f'  In driver terms: MAE of {MAE} trips / {TRIPS_PER_DRIVER} trips per driver')
print(f'  = roughly {MAE/TRIPS_PER_DRIVER:.0f} drivers per zone per hour typical error')
print()
print('For an operations manager:')
print('  If we forecast 20 drivers needed in Bole at 6pm,')
print('  the true need is typically between 15 and 25 -- an actionable planning range.')